# Avance 02 · Kaggle · Etapa 2: perfil y validación de calidad
**Equipo A&C | Hechos de tránsito en la CDMX**

## Propósito
Partimos de los **43,217 registros aptos** de la Etapa 1 (2018–2022, 16 alcaldías, 45 columnas originales). **No eliminamos filas ni sobrescribimos el origen**. Trabajamos con copias, detectamos posibles problemas y guardamos evidencias reproducibles. 

**Precaución:** Kaggle reproduce datos ATUS; sus registros pueden solaparse con INEGI–ATUS. La igualdad de totales anuales no prueba identidad fila por fila y **no debemos sumar ambas fuentes como accidentes independientes**.

En esta etapa **detectamos** duplicados exactos, pero **no los retiramos**: antes debemos confirmar la unidad de observación y las reglas de la Etapa 3.

## 1. Localizamos el CSV apto de la Etapa 1
Buscamos el archivo exportado por nuestro notebook anterior, sin depender de que el notebook esté en la misma carpeta. Si Anaconda no lo encuentra, indicamos la ruta manualmente. Leemos **todo como texto** para conservar claves con ceros a la izquierda y evitar conversiones involuntarias.

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import display

# Si no encontramos el archivo automáticamente, escribimos su ruta completa.
RUTA_APTO_MANUAL = None
NOMBRE = 'Kaggle_ATUS_etapa1.csv'

if RUTA_APTO_MANUAL is not None:
    RUTA_APTO = Path(RUTA_APTO_MANUAL).expanduser().resolve()
else:
    # Probamos las ubicaciones habituales de salida de la Etapa 1.
    candidatos = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        for ruta in [base / 'resultados' / 'resultados_Kaggle' / 'etapa1' / NOMBRE,
                     base / 'resultados_Kaggle' / 'etapa1' / NOMBRE,
                     base / NOMBRE]:
            if ruta.is_file():
                candidatos.append(ruta.resolve())
    candidatos = list(dict.fromkeys(candidatos))
    if len(candidatos) != 1:
        print('Rutas encontradas:', *candidatos, sep='\n - ')
        raise FileNotFoundError('Indicamos la ubicación exacta en RUTA_APTO_MANUAL.')
    RUTA_APTO = candidatos[0]

# Generamos una carpeta nueva, hermana de etapa1, para no mezclar evidencias.
SALIDA = RUTA_APTO.parent.parent / 'etapa2'
SALIDA.mkdir(parents=True, exist_ok=True)

# Leemos el CSV de la Etapa 1 y creamos una copia de trabajo.
datos_etapa1 = pd.read_csv(RUTA_APTO, dtype=str, encoding='utf-8-sig', index_col=False)
datos_trabajo = datos_etapa1.copy(deep=True)
COLUMNAS_ORIGINALES = [c for c in datos_etapa1.columns if c not in {
    'archivo_origen', 'fila_origen', 'anio_normalizado',
    'entidad_normalizada', 'municipio_normalizado',
    'motivo_exclusion', 'alcaldia_normalizada'
}]
print('Archivo leído:', RUTA_APTO)
print('Filas:', len(datos_trabajo), '| Columnas originales:', len(COLUMNAS_ORIGINALES))
print('Carpeta de evidencias:', SALIDA)
display(datos_trabajo.head(3))

Archivo leído: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa1/Kaggle_ATUS_etapa1.csv
Filas: 43217 | Columnas originales: 45
Carpeta de evidencias: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_Kaggle/etapa2


,COBERTURA,ID_ENTIDAD,ID_MUNICIPIO,ANIO,MES,ID_HORA,ID_MINUTO,ID_DIA,DIASEMANA,URBANA,...,NEHERIDO,CLASACC,ESTATUS,archivo_origen,fila_origen,anio_normalizado,entidad_normalizada,municipio_normalizado,motivo_exclusion,alcaldia_normalizada
0,Municipal,09,002,2018,01,7,0,17,Miércoles,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75450,2018,09,002,apto,Azcapotzalco
1,Municipal,09,002,2018,01,21,0,17,Miércoles,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75451,2018,09,002,apto,Azcapotzalco
2,Municipal,09,002,2018,01,16,46,21,Domingo,Accidente en intersección,...,0,Sólo daños,Cifras Definitivas,atus_anual_2018.csv,75452,2018,09,002,apto,Azcapotzalco


## 2. Construimos nuestro perfil inicial de calidad
Contamos faltantes, cadenas vacías, valores distintos y tipos de lectura por columna. Este diagnóstico **no cambia** los datos. También revisamos que la selección inicial conserve los años y las claves geográficas esperadas.

In [2]:
# Creamos una copia para normalizar únicamente las comprobaciones.
perfil_trabajo = datos_trabajo.copy(deep=True)

perfil = []
for columna in COLUMNAS_ORIGINALES:
    serie = perfil_trabajo[columna].astype('string')
    vacio = serie.isna() | serie.str.strip().eq('').fillna(False)
    perfil.append({
        'columna': columna,
        'tipo_lectura': str(perfil_trabajo[columna].dtype),
        'faltantes_o_vacios': int(vacio.sum()),
        'porcentaje_faltantes': round(float(vacio.mean() * 100), 2),
        'valores_distintos_no_nulos': int(serie.dropna().str.strip().nunique())
    })
perfil_df = pd.DataFrame(perfil)
display(perfil_df)
perfil_df.to_csv(SALIDA / 'perfil_columnas_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

# Normalizamos en columnas auxiliares, sin modificar las columnas originales.
validacion = datos_trabajo.copy(deep=True)
validacion['anio_revision'] = pd.to_numeric(validacion['ANIO'].astype('string').str.strip(), errors='coerce')
validacion['entidad_revision'] = validacion['ID_ENTIDAD'].astype('string').str.strip().str.zfill(2)
validacion['municipio_revision'] = validacion['ID_MUNICIPIO'].astype('string').str.strip().str.zfill(3)

print('Años:', sorted(validacion['anio_revision'].dropna().unique().tolist()))
print('Entidades:', sorted(validacion['entidad_revision'].dropna().unique().tolist()))
print('Municipios:', sorted(validacion['municipio_revision'].dropna().unique().tolist()))

,columna,tipo_lectura,faltantes_o_vacios,porcentaje_faltantes,valores_distintos_no_nulos
0,COBERTURA,object,0,0.0,1
1,ID_ENTIDAD,object,0,0.0,1
2,ID_MUNICIPIO,object,0,0.0,16
3,ANIO,object,0,0.0,5
4,MES,object,0,0.0,12
5,ID_HORA,object,0,0.0,34
6,ID_MINUTO,object,0,0.0,70
7,ID_DIA,object,0,0.0,40
8,DIASEMANA,object,0,0.0,10
9,URBANA,object,0,0.0,3


Años: [2018, 2019, 2020, 2021, 2022]
Entidades: ['09']
Municipios: ['002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017']


## 3. Identificamos posibles duplicados exactos
Comparamos **las 45 columnas originales**, no las columnas añadidas durante la limpieza (`archivo_origen`, `fila_origen`, etc.). Esto permite detectar filas idénticas en su contenido de origen. **No retiramos ninguna**: dos registros iguales pueden representar observaciones distintas si la fuente no incluye un identificador único confiable.

In [3]:
# Creamos otra copia independiente para estudiar duplicados.
revision_duplicados = datos_trabajo[COLUMNAS_ORIGINALES].copy(deep=True)

# Marcamos todas las filas que pertenecen a un grupo duplicado.
marca_grupo = revision_duplicados.duplicated(keep=False)
# Marcamos solo las repeticiones adicionales, conservando la primera como referencia.
marca_adicional = revision_duplicados.duplicated(keep='first')

resumen_duplicados_df = pd.DataFrame([{
    'filas_recibidas': len(revision_duplicados),
    'filas_en_grupos_duplicados': int(marca_grupo.sum()),
    'repeticiones_adicionales_exactas': int(marca_adicional.sum()),
    'columnas_comparadas': len(COLUMNAS_ORIGINALES),
    'filas_eliminadas': 0
}])
display(resumen_duplicados_df)
resumen_duplicados_df.to_csv(SALIDA / 'resumen_duplicados_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

# Guardamos evidencia con procedencia, sin eliminar ni alterar observaciones.
evidencia_duplicados = datos_trabajo.loc[marca_grupo].copy()
evidencia_duplicados['duplicado_adicional_exacto'] = marca_adicional.loc[marca_grupo].to_numpy()
evidencia_duplicados.to_csv(SALIDA / 'evidencia_duplicados_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

,filas_recibidas,filas_en_grupos_duplicados,repeticiones_adicionales_exactas,columnas_comparadas,filas_eliminadas
0,43217,76,38,45,0


## 4. Validamos fecha, hora y calendario
ATUS separa la fecha en `ANIO`, `MES` e `ID_DIA`, y la hora en `ID_HORA` e `ID_MINUTO`. Convertimos **en columnas nuevas**. Los valores faltantes o códigos especiales quedan registrados como inválidos o no interpretables; no inventamos fechas ni reemplazamos silenciosamente valores originales.

La validación de `ID_HORA` y `ID_MINUTO` usa rangos de reloj convencionales; si existen códigos de «no especificado», los reportamos como no interpretables y los documentamos.

In [4]:
# Seguimos trabajando con otra copia para preservar el DataFrame de lectura.
revision_temporal = validacion.copy(deep=True)
for original, nueva in [('MES','mes_revision'), ('ID_DIA','dia_revision'),
                        ('ID_HORA','hora_revision'), ('ID_MINUTO','minuto_revision')]:
    revision_temporal[nueva] = pd.to_numeric(
        revision_temporal[original].astype('string').str.strip(), errors='coerce')

# Construimos una fecha sin modificar ANIO, MES ni ID_DIA.
revision_temporal['fecha_revision'] = pd.to_datetime(
    pd.DataFrame({
        'year': revision_temporal['anio_revision'],
        'month': revision_temporal['mes_revision'],
        'day': revision_temporal['dia_revision']
    }), errors='coerce')
revision_temporal['fecha_invalida'] = revision_temporal['fecha_revision'].isna()
revision_temporal['hora_no_interpretable'] = ~revision_temporal['hora_revision'].between(0, 23)
revision_temporal['minuto_no_interpretable'] = ~revision_temporal['minuto_revision'].between(0, 59)

resumen_temporal_df = pd.DataFrame([{
    'filas': len(revision_temporal),
    'fechas_invalidas': int(revision_temporal['fecha_invalida'].sum()),
    'horas_no_interpretables': int(revision_temporal['hora_no_interpretable'].sum()),
    'minutos_no_interpretables': int(revision_temporal['minuto_no_interpretable'].sum())
}])
display(resumen_temporal_df)
resumen_temporal_df.to_csv(SALIDA / 'resumen_temporal_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

# Guardamos evidencia solo de las filas que requieren revisión temporal.
marcas_temporales = (revision_temporal['fecha_invalida'] |
                     revision_temporal['hora_no_interpretable'] |
                     revision_temporal['minuto_no_interpretable'])
columnas_evidencia = ['archivo_origen', 'fila_origen', 'ANIO', 'MES', 'ID_DIA',
                      'ID_HORA', 'ID_MINUTO', 'fecha_invalida',
                      'hora_no_interpretable', 'minuto_no_interpretable']
revision_temporal.loc[marcas_temporales, columnas_evidencia].copy().to_csv(
    SALIDA / 'evidencia_temporal_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

,filas,fechas_invalidas,horas_no_interpretables,minutos_no_interpretables
0,43217,0,0,0


## 5. Revisamos las variables de personas fallecidas y heridas
En ATUS existen columnas separadas para distintas clases de víctimas. Verificamos valores no numéricos, negativos y posibles inconsistencias. **No calculamos letalidad ni inferimos causalidad**. Los totales derivados sirven para el control de calidad y no sustituyen la definición de indicadores del proyecto.

**Importante:** no sumamos las columnas `COND*`, `PASA*`, `PEAT*`, `CICL*`, `OTRO*`, `NE*` para obtener automáticamente un total validado sin revisar el diccionario de ATUS. Aquí solo evaluamos si cada campo se puede interpretar como un conteo no negativo.

In [5]:
# Identificamos exclusivamente las variables de víctimas presentes en el CSV.
COLUMNAS_VICTIMAS = [
    'CONDMUERTO', 'CONDHERIDO', 'PASAMUERTO', 'PASAHERIDO',
    'PEATMUERTO', 'PEATHERIDO', 'CICLMUERTO', 'CICLHERIDO',
    'OTROMUERTO', 'OTROHERIDO', 'NEMUERTO', 'NEHERIDO'
]
COLUMNAS_VICTIMAS = [c for c in COLUMNAS_VICTIMAS if c in datos_trabajo.columns]

revision_victimas = datos_trabajo.copy(deep=True)
resumen_victimas = []
for columna in COLUMNAS_VICTIMAS:
    texto = revision_victimas[columna].astype('string').str.strip()
    numero = pd.to_numeric(texto, errors='coerce')
    vacio = texto.isna() | texto.eq('').fillna(False)
    no_numerico = numero.isna() & ~vacio
    negativo = numero.lt(0).fillna(False)
    no_entero = (numero.notna() & numero.mod(1).ne(0)).fillna(False)
    resumen_victimas.append({
        'variable': columna,
        'faltantes_o_vacios': int(vacio.sum()),
        'no_numericos': int(no_numerico.sum()),
        'negativos': int(negativo.sum()),
        'no_enteros': int(no_entero.sum()),
        'maximo_observado': numero.max()
    })
resumen_victimas_df = pd.DataFrame(resumen_victimas)
display(resumen_victimas_df)
resumen_victimas_df.to_csv(SALIDA / 'resumen_victimas_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

,variable,faltantes_o_vacios,no_numericos,negativos,no_enteros,maximo_observado
0,CONDMUERTO,0,0,0,0,8
1,CONDHERIDO,0,0,0,0,4
2,PASAMUERTO,0,0,0,0,4
3,PASAHERIDO,0,0,0,0,42
4,PEATMUERTO,0,0,0,0,2
5,PEATHERIDO,0,0,0,0,7
6,CICLMUERTO,0,0,0,0,2
7,CICLHERIDO,0,0,0,0,2
8,OTROMUERTO,0,0,0,0,1
9,OTROHERIDO,0,0,0,0,2


## 6. Comprobamos las claves geográficas y conservamos las 16 alcaldías
Revisamos que cada registro corresponda a la entidad `09` y a una de las claves municipales `002`–`017`. No exigimos que haya la misma cantidad de hechos en cada alcaldía. Una discrepancia sería una alerta para revisar la Etapa 1, no un motivo para borrar filas automáticamente.

In [6]:
# Creamos copia de las columnas geográficas para una validación independiente.
revision_geo = validacion[['ID_ENTIDAD', 'ID_MUNICIPIO', 'entidad_revision',
                           'municipio_revision']].copy(deep=True)
claves_validas = {f'{i:03d}' for i in range(2, 18)}
revision_geo['entidad_invalida'] = ~revision_geo['entidad_revision'].eq('09')
revision_geo['municipio_invalido'] = ~revision_geo['municipio_revision'].isin(claves_validas)

resumen_geo_df = pd.DataFrame([{
    'filas': len(revision_geo),
    'entidades_invalidas': int(revision_geo['entidad_invalida'].sum()),
    'municipios_invalidos': int(revision_geo['municipio_invalido'].sum()),
    'alcaldias_distintas_presentes': int(revision_geo['municipio_revision'].nunique())
}])
display(resumen_geo_df)
resumen_geo_df.to_csv(SALIDA / 'resumen_geografico_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

,filas,entidades_invalidas,municipios_invalidos,alcaldias_distintas_presentes
0,43217,0,0,16


## 7. Construimos el resumen de impacto y ejecutamos comprobaciones
La Etapa 2 es **diagnóstica**: no retiramos registros. Las comprobaciones confirman la integridad del proceso, **no** que los datos carezcan de problemas. Los hallazgos se documentan para decidir reglas justificadas en la Etapa 3.

In [7]:
impacto_df = pd.DataFrame([{
    'filas_recibidas_etapa2': len(datos_etapa1),
    'filas_retiradas_etapa2': 0,
    'filas_conservadas_etapa2': len(datos_trabajo),
    'repeticiones_adicionales_exactas_detectadas': int(marca_adicional.sum()),
    'fechas_invalidas_detectadas': int(revision_temporal['fecha_invalida'].sum())
}])

comprobaciones = {
    'total_esperado_etapa1': len(datos_etapa1) == 43217,
    'sin_eliminaciones': len(datos_trabajo) == len(datos_etapa1),
    'columnas_originales_45': len(COLUMNAS_ORIGINALES) == 45,
    'solo_anios_2018_2022': validacion['anio_revision'].between(2018, 2022).all(),
    'solo_entidad_09': validacion['entidad_revision'].eq('09').all(),
    'solo_16_alcaldias': validacion['municipio_revision'].isin(claves_validas).all(),
    'conteo_duplicados_consistente': int(marca_adicional.sum()) <= int(marca_grupo.sum()),
    'datos_de_lectura_sin_cambios': datos_etapa1.equals(datos_trabajo),
}
comprobaciones_df = pd.DataFrame([
    {'comprobacion': nombre, 'cumple': bool(valor)}
    for nombre, valor in comprobaciones.items()
])

display(impacto_df)
display(comprobaciones_df)
impacto_df.to_csv(SALIDA / 'impacto_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_Kaggle_etapa2.csv', index=False, encoding='utf-8-sig')

# Exportamos la tabla apta intacta como evidencia de conservación de registros.
# La Etapa 3 trabajará sobre esta copia, nunca sobre el CSV nacional original.
DATOS_ETAPA2 = SALIDA / 'Kaggle_ATUS_etapa2_sin_eliminaciones.csv'
datos_trabajo.to_csv(DATOS_ETAPA2, index=False, encoding='utf-8-sig')

if not comprobaciones_df['cumple'].all():
    print('Revisamos las comprobaciones marcadas como False:')
    display(comprobaciones_df.loc[~comprobaciones_df['cumple']].copy())
    raise AssertionError('La Etapa 2 requiere revisar las comprobaciones.')
print('Etapa 2 Kaggle completada: comprobaciones de integridad superadas.')

,filas_recibidas_etapa2,filas_retiradas_etapa2,filas_conservadas_etapa2,repeticiones_adicionales_exactas_detectadas,fechas_invalidas_detectadas
0,43217,0,43217,38,0


,comprobacion,cumple
0,total_esperado_etapa1,True
1,sin_eliminaciones,True
2,columnas_originales_45,True
3,solo_anios_2018_2022,True
4,solo_entidad_09,True
5,solo_16_alcaldias,True
6,conteo_duplicados_consistente,True
7,datos_de_lectura_sin_cambios,True


Etapa 2 Kaggle completada: comprobaciones de integridad superadas.


## 8. Interpretamos los hallazgos y preparamos la bitácora

**Nuestro procedimiento:**
1. Leímos la tabla delimitada de Kaggle y creamos copias independientes.
2. Perfilamos las 45 columnas originales y cuantificamos valores faltantes.
3. Detectamos duplicados exactos, sin asumir que representan errores.
4. Validamos fechas, horas, minutos, claves geográficas y conteos de víctimas.
5. Conservamos todas las filas y exportamos evidencia de cada revisión.

**Decisión para la Etapa 3:** estudiaremos las observaciones marcadas y justificaremos cualquier regla. **No eliminaremos duplicados solo por ser idénticos** y evitaremos mezclar Kaggle con INEGI como si fueran sucesos distintos.

**Archivos generados:** perfil de columnas, duplicados, evidencia de duplicados, resumen temporal, evidencia temporal, resumen de víctimas, resumen geográfico, impacto, comprobaciones y CSV conservado.

**Para compartir resultados:** copiamos las tablas `resumen_duplicados_df`, `resumen_temporal_df`, `resumen_victimas_df`, `resumen_geo_df`, `impacto_df` y `comprobaciones_df`.